# Simple and Effective Masked Diffusion Language Models

This notebook walks through the key components of the codebase implementing **Simple and Effective Masked Diffusion Language Models (MDLMs)**. Each section pairs concise explanations with the equations embodied by the source code so readers can understand how training, parameterization, and sampling fit together.

## 1. Configuration snapshot
The project is configured via YAML files and Hydra. The top-level configuration (`configs/config.yaml`) wires together the tokenizer, backbone, noise schedule, optimization, and evaluation settings. Below we preview the default configuration to ground the rest of the discussion.

In [ ]:
import yaml, pathlib
config_path = pathlib.Path('configs/config.yaml')
with config_path.open() as f:
    cfg = yaml.safe_load(f)
cfg


## 2. Forward (corruption) process
MDLMs mask tokens forward in time. For an input $x_0$ and per-example mask rate $m$, the corrupted sample $x_t$ is drawn independently per position:

$$x_t^{(i)} = egin{cases}	exttt{[MASK]}, & 	ext{if } u < m \ x_0^{(i)}, & 	ext{otherwise}\end{cases} \quad 	ext{with } u \sim \mathcal{U}(0,1).$$

In code, `move_chance` is computed from the noise schedule and applied inside `Diffusion.q_xt` to replace tokens with the mask index. The same primitive is reused for discrete-time schedules (D3PM) and continuous-time SUBS/SEDD parameterizations.

In [ ]:
import torch

def example_corruption(x0, move_prob, mask_index=0):
    move_prob = torch.tensor(move_prob)[:, None]
    # Mirror the q_xt logic without instantiating the full LightningModule
    move_indices = torch.rand_like(x0.float()) < move_prob
    return torch.where(move_indices, mask_index, x0)

x0 = torch.tensor([[1, 2, 3, 4]])
example_corruption(x0, move_prob=[0.3])


## 3. Noise schedules
Time $t \in [0,1]$ maps to a total noise level $\sigma(t)$ and its rate $g(t)$ (see `noise_schedule.py`). Common options include:

* **Log-linear**: $\sigma(t) = -\log(1 - (1 - arepsilon)t)$ with $g(t) = rac{1-arepsilon}{1-(1-arepsilon)t}$.
* **Cosine**: $\sigma(t) = -\log(arepsilon + (1-arepsilon)\cos(	frac{\pi t}{2}))$ and $g(t) = 	frac{\pi}{2}	frac{(1-arepsilon)\sin(	frac{\pi t}{2})}{(1-arepsilon)\cos(	frac{\pi t}{2}) + arepsilon}$.
* **Linear/Geometric**: simple linear or log-space interpolations between $\sigma_	ext{min}$ and $\sigma_	ext{max}$.

The total noise determines the masking probability via $m = 1 - e^{-\sigma(t)}$ (or its discrete-time analogue), which drives the forward corruption step above.

## 4. Denoising parameterizations
The model outputs log-scores over the vocabulary conditioned on noisy input $x_t$ and time conditioning $\sigma$ (or $t$). Three parameterizations correspond to different objectives in the paper:

1. **SUBS (subspace score matching)** — enforces probabilities only where masking can occur. After normalizing logits, masked positions receive a shift $\log k$ with $k = rac{e^{-\sigma}}{1-e^{-\sigma}}$, while unmasked positions disallow other tokens by setting their logits to $-\infty$.
2. **D3PM (discrete diffusion)** — treats logits as a categorical distribution over replacements; optionally forbids masks when substitution masking is enabled.
3. **SEDD (score entropy of discrete data)** — scales logits by $\log\!\left(	frac{e^{\sigma}-1}{|V|-1}ight)$ and sets the log-score of the observed token to zero, matching the entropy objective on masked sites.

## 5. Training objectives
### Continuous-time SUBS loss
For continuous SUBS, the objective uses the log-probability assigned to the clean token $x_0$ scaled by the noise derivative:

$$\mathcal{L}_{\text{SUBS}} = -\,\frac{\mathrm{d}\sigma}{\mathrm{d}t}\;\frac{\log p_\theta(x_0 \mid x_t, \sigma)}{e^{\sigma}-1}.\n$$

Optional importance sampling or change-of-variables reweights this term using $\log(1 - e^{-\sigma_\text{min}})$.

### Discrete-time D3PM
When `T>0`, time is quantized and the variational bound from `Diffusion._d3pm_loss` is applied at masked positions:

$$\mathcal{L}_{\text{VB}} = T \Big[ \tfrac{\Delta t}{t}\big(\log(\tfrac{\alpha_t p_\theta([MASK])}{t}+1) - \log p_\theta(x_0)\big) + (1-\tfrac{\Delta t}{t})\big(\log(\tfrac{\alpha_{t-\Delta t} p_\theta([MASK])}{t-\Delta t}+1) - \log(\tfrac{\alpha_t p_\theta([MASK])}{t}+1)\big) \Big].$$

A reconstruction term adds $-\log p_\theta(x_0 \mid t{=}0)$ when using the D3PM parameterization; SUBS omits it because masked and unmasked logits already align.

### SEDD score entropy
SEDD minimizes the entropy of the predicted score distribution on masked tokens. For positions $i$ where $x_t^{(i)}$ is masked, with $q = \tfrac{1}{e^{\sigma}-1}$ and $s_y = \exp(\log s_\theta(y \mid x_t, \sigma))$, the per-token loss is

$$\mathcal{L}_{\text{SEDD}}^{(i)} = q\,(\log q - 1) + \sum_{y \neq \texttt{[MASK]}} s_y - q\,\log s_{x_0^{(i)}}.\n$$

Unmasked positions incur zero loss, mirroring the behavior in `_score_entropy`.


## 6. Sampling
Generation starts from a fully masked prior and repeatedly denoises:

1. **Time grid**: pick $t_0=1 > t_1 > \dots > t_S \approx 0$ with step size $\Delta t$.
2. **Predict clean tokens**: obtain $p_\theta(x_0 \mid x_t, \sigma_t)$ and derive a transition distribution $q(x_{t-\Delta t} \mid x_t)$ using either the analytic transport update or a DDPM-style categorical sampler.
3. **Optional noise removal**: at the final step, apply the denoiser update to project remaining probability mass away from the mask token.

The autoregressive fallback (`parameterization == 'ar'`) instead samples token-by-token using cached Gumbel noise for efficiency.

In [ ]:
# Pseudocode sketch mirroring Diffusion._sample
def masked_diffusion_sampling(model, steps=10, eps=1e-5):
    x = torch.full((1, model.config.model.length), model.mask_index)
    t_grid = torch.linspace(1, eps, steps + 1)
    dt = (1 - eps) / steps
    for i, t in enumerate(t_grid[:-1]):
        sigma = model.noise(t)[0]
        if model.sampler == 'analytic':
            x = model._analytic_update(x, t_grid[i], dt)
        else:
            x = model._ddpm_update(x, t_grid[i], dt)
    return x

# Sampling requires a trained Lightning checkpoint; the above highlights the update logic.


## 7. Takeaways
* Masked corruption is driven by the noise schedule $\sigma(t)$, giving a direct link between continuous-time diffusion and token masking.
* Parameterizations tailor the log-score manipulation to the desired objective (SUBS, D3PM, or SEDD), but all share the same masking-centric forward process.
* Sampling mirrors the forward process in reverse, using either analytic transport or DDPM-style categorical updates to progressively unmask tokens.